# 04 - Helpful-Review and Temporal Trend Analysis
## Objectives 3 and 4

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

**Objective 3** surfaces the most-voted reviews and measures helpfulness against star rating.

**Objective 4** tracks review volume and rating across the full 1999 to 2023 span. Older reviews
have had far longer to accumulate votes, so helpfulness and year cannot be read independently of
each other.

In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 04 - Objectives 3 and 4")
spark = get_spark("04 helpfulness and temporal")
df = load_analytical(spark).cache()
n_clean = df.count()
print(f"analytical dataset: {n_clean:,} rows")

### Objective 3 - Helpful-review analysis

In [ ]:
top_helpful = (df.select("helpful_vote", "rating", "Category", "review_length",
                         F.substring("title", 1, 45).alias("title"),
                         F.substring("text", 1, 60).alias("text_preview"))
               .orderBy(F.desc("helpful_vote")).limit(10))
print("Top 10 most-helpful reviews"); top_helpful.show(truncate=False)

help_by_rating = (df.groupBy("rating")
                  .agg(F.round(F.avg("helpful_vote"), 3).alias("avg_helpful"),
                       F.max("helpful_vote").alias("max_helpful"),
                       F.round(100 * F.avg((F.col("helpful_vote") >= 1).cast("int")), 1).alias("pct_with_vote"),
                       F.round(100 * F.avg("is_highly_helpful"), 2).alias("pct_highly_helpful"))
                  .orderBy("rating"))
oh = df.agg(F.round(F.avg("helpful_vote"), 3).alias("avg"),
            F.round(100 * F.avg((F.col("helpful_vote") >= 1).cast("int")), 1).alias("pct_any")).first()
print(f"Overall: mean {oh['avg']} votes, {oh['pct_any']}% of reviews receive at least one")
help_by_rating.show()

save_table(top_helpful, "obj3_top_helpful_reviews")
save_table(help_by_rating, "obj3_helpful_by_rating")
save_table(pd.DataFrame([{"mean_helpful_vote": oh["avg"], "pct_with_any_vote": oh["pct_any"]}]),
           "obj3_helpful_summary");

In [ ]:
hbr = help_by_rating.toPandas()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(hbr["rating"].astype(int).astype(str), hbr["avg_helpful"], color=ACCENT)
ax[0].set_title("Average helpful votes by star rating")
ax[0].set_xlabel("Stars"); ax[0].set_ylabel("Average helpful votes")
for i, v in enumerate(hbr["avg_helpful"]):
    ax[0].text(i, v, f"{v}", ha="center", va="bottom", fontsize=9)

ax[1].bar(hbr["rating"].astype(int).astype(str), hbr["pct_with_vote"], color="#f59e0b")
ax[1].set_title("Share receiving at least one helpful vote")
ax[1].set_xlabel("Stars"); ax[1].set_ylabel("% with a helpful vote")
for i, v in enumerate(hbr["pct_with_vote"]):
    ax[1].text(i, v, f"{v}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
savefig(fig, "fig04_helpful_reviews", "Objective 3 - helpfulness by star rating")
plt.show()

### Objective 4 - Temporal trend analysis

In [ ]:
yearly = (df.groupBy("review_year")
          .agg(F.count("*").alias("reviews"),
               F.round(F.avg("rating"), 3).alias("avg_rating"),
               F.round(F.avg("helpful_vote"), 3).alias("avg_helpful"),
               F.round(F.avg("review_length"), 1).alias("avg_length"))
          .orderBy("review_year"))
yearly.show(40, truncate=False)

n_surge = df.filter(F.col("review_year").between(2019, 2021)).count()
n_before = df.filter(F.col("review_year").between(2016, 2018)).count()
peak = yearly.orderBy(F.desc("reviews")).first()
print(f"2019-2021: {n_surge:,} vs 2016-2018: {n_before:,} (x{n_surge/max(n_before,1):.1f})")
print(f"Busiest year: {peak['review_year']} with {peak['reviews']:,} reviews")

save_table(yearly, "obj4_yearly_trend")
save_table(pd.DataFrame([{"peak_year": int(peak["review_year"]), "peak_reviews": int(peak["reviews"]),
                          "reviews_2019_2021": n_surge, "reviews_2016_2018": n_before,
                          "surge_multiple": round(n_surge / max(n_before, 1), 2)}]),
           "obj4_trend_summary");

In [ ]:
yr = yearly.toPandas()
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(yr["review_year"], yr["reviews"], color="#93c5fd", label="Review volume")
ax1.set_xlabel("Year"); ax1.set_ylabel("Number of reviews", color="#1d4ed8")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax1.tick_params(axis="y", labelcolor="#1d4ed8")

ax2 = ax1.twinx()
ax2.plot(yr["review_year"], yr["avg_rating"], color="#dc2626", marker="o", lw=2)
ax2.set_ylabel("Average rating", color="#dc2626")
ax2.set_ylim(max(1, yr["avg_rating"].min() - 0.5), min(5.05, yr["avg_rating"].max() + 0.5))
ax2.tick_params(axis="y", labelcolor="#dc2626"); ax2.grid(False)
ax2.axvspan(2018.5, 2021.5, color="#fde68a", alpha=0.35)

plt.title("Objective 4 - review volume and average rating over time")
fig.tight_layout()
savefig(fig, "fig05_temporal_trend", "Objective 4 - volume and rating over time")
plt.show()

In [ ]:
cat_year = (df.filter(F.col("review_year") >= 2015)
            .groupBy("review_year", "Category").count().orderBy("review_year"))
cy = cat_year.toPandas()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
for cat in CATS:
    sub = cy[cy["Category"] == cat]
    if len(sub):
        ax[0].plot(sub["review_year"], sub["count"], marker="o", label=cat, color=ccol(cat), lw=2)
ax[0].set_title("Review volume by category since 2015")
ax[0].set_xlabel("Year"); ax[0].set_ylabel("Reviews")
ax[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax[0].legend()

ax[1].plot(yr["review_year"], yr["avg_helpful"], marker="o", color="#7c3aed", lw=2)
ax[1].set_title("Average helpful votes by review year")
ax[1].set_xlabel("Year"); ax[1].set_ylabel("Average helpful votes")
ax[1].annotate("older reviews have had\nlonger to collect votes",
               xy=(0.42, 0.72), xycoords="axes fraction", fontsize=9, color="#4c1d95")

plt.tight_layout()
savefig(fig, "fig06_category_and_vote_trend", "Objective 4 - category volume and vote-age effect")
plt.show()

save_table(cat_year, "obj4_category_yearly");

### Findings

Helpful votes concentrate in the low-star tail rather than spreading evenly, which fits the idea
that a critical review does more work for a shopper than another five-star endorsement.

On the temporal side, read the volume curve and the helpful-vote curve together. Average votes per
review fall steadily across the years, and that is largely an age effect rather than a drop in
review quality: a 2005 review has had eighteen extra years to collect votes that a 2023 review has
not had. Any cross-year comparison of helpfulness needs that caveat attached.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")